# DuckDB

A refresher on **DuckDB** — an **in-process, columnar OLAP SQL database**. Think "SQLite for
analytics": a single embedded library, no server, but built for fast aggregations and scans
over big tables instead of single-row transactions. It runs SQL directly over Parquet/CSV/JSON
files and over your in-memory pandas/Polars/Arrow frames with zero copy. For the DataFrame-API
take on the same columnar/Arrow world see [[polars]]; for the classic in-memory workflow see
[[numpy-pandas-scipy]].

**Domain:** Data Analysis & Research  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** DuckDB is a relational database that runs **inside your process** — `import
duckdb` and you have a full SQL engine, no server to install, connect to, or keep alive. Unlike
SQLite (which is row-oriented and tuned for transactional `OLTP` workloads), DuckDB stores and
processes data **column-by-column** with a **vectorized** execution engine, which is exactly the
shape analytical queries want: scan a few columns of many rows, filter, group, aggregate, join.

**The problem it solves.** You have a few GB of Parquet/CSV and you want to `GROUP BY` and join
it. Spinning up Postgres/Spark is heavy operational overhead; pandas pins one core and may not
fit the data in RAM; writing the aggregation by hand in Python is slow and verbose. DuckDB lets
you point SQL straight at the files — `SELECT ... FROM 'data/*.parquet'` — with a real query
optimizer, multi-threaded execution, and **larger-than-memory** spill-to-disk, all from a pip
install. It reads only the columns and row-groups it needs and routinely beats pandas by an
order of magnitude on group-bys and joins.

**When to reach for it.** Interactive analytics and ETL on local/medium data (MBs to ~100s of
GB); querying Parquet/CSV/JSON lakes without loading them first; SQL-shaped transforms inside a
Python pipeline; a fast local stand-in for a warehouse. **When not to:** high-concurrency
transactional workloads with many small writes (that's Postgres/SQLite territory); a shared
multi-user database with concurrent writers (DuckDB is single-writer, embedded); or petabyte
distributed jobs that genuinely need a cluster (Spark/Trino/BigQuery).

## 2. Mental Model

**DuckDB is "SQLite for analytics": one embedded file/library, but columnar and vectorized so it
chews through aggregations instead of point lookups. And it treats your DataFrames and your
Parquet/CSV files as if they were already tables — point SQL at them in place, no load step.**

Two ideas make it click:

1. **Row store vs column store.** SQLite/Postgres store a row's fields together — great for
   "fetch this one order," bad for "average every order's total" because you drag every column
   through cache to read one. DuckDB stores each column contiguously and processes it in
   **vectors** (~2048 values at a time) through tight CPU loops, so a `SUM`/`GROUP BY` touches
   only the columns it needs and stays cache- and SIMD-friendly.

2. **The data is already the table.** DuckDB has **replacement scans**: a pandas/Polars/Arrow
   variable in your Python scope, or a path like `'sales/*.parquet'`, can appear *directly* in
   the `FROM` clause. There's no import/ingest phase — DuckDB reads the external data lazily,
   pushing filters and column projection down into the scan. So the engine, not your Python, does
   the heavy lifting, and nothing gets copied unnecessarily.

If pandas is "load everything into memory, then operate," DuckDB is "describe the result in SQL
and let an optimizer read only what it needs, in parallel, possibly spilling to disk."

## 3. Key Concepts

- **In-process / embedded.** No server. `duckdb.connect()` (or the module-level default
  connection) gives you an engine in your own process. `connect("file.db")` persists to a single
  file; `connect()` / `:memory:` is ephemeral.
- **Columnar + vectorized.** Storage is column-major; execution runs batches of ~2048 values
  through vectorized operators. This is *why* analytics are fast and why row-at-a-time point
  lookups are comparatively unremarkable.
- **Replacement scans.** A DataFrame variable or a file glob can be used *as a table* in `FROM`
  with no explicit load: `SELECT * FROM my_df`, `SELECT * FROM 'data/*.parquet'`. Filters and
  projections are pushed down into the scan.
- **Zero-copy DataFrame interop.** DuckDB speaks Apache Arrow, so it shares memory with pandas,
  Polars, and Arrow without serializing. `.df()` → pandas, `.pl()` → Polars, `.arrow()` → Arrow,
  `.fetchall()` → list of tuples.
- **Relational API.** Besides raw SQL strings you can chain Python methods —
  `con.table("t").filter("x > 1").aggregate("k, sum(v)")` — building a lazy relation that only
  executes when you materialize it (`.df()`, `.fetchall()`, `.show()`).
- **Out-of-core / larger-than-memory.** With a persistent database (or a temp dir set) DuckDB
  spills intermediate results to disk, so joins/aggregations can exceed RAM. Tune with
  `SET memory_limit` and `SET threads`.
- **`read_parquet` / `read_csv_auto` / `read_json_auto`.** Table functions that infer schema and
  read files directly; globs and Hive-partitioned directories are supported.
- **Extensions.** Loadable add-ons: `httpfs` (read S3/HTTP), `parquet`, `json`, `spatial`,
  `fts` (full-text), `iceberg`/`delta`. Install with `INSTALL x; LOAD x;`.

## 4. Setup

DuckDB is a single self-contained wheel — the engine is bundled C++, no server and no system
dependencies. CPU-only, installs in seconds. The optional `httpfs` extension (for S3/HTTP) and
others are downloaded on first `INSTALL`.

In [ ]:
# %pip install duckdb
import duckdb

print("duckdb", duckdb.__version__)

# The simplest possible query — no connection object needed; there's a default in-memory one.
print(duckdb.sql("SELECT 42 AS answer, 'duck' AS bird").fetchall())

## 5. Worked Examples

A tiny in-memory sales table is enough to show the whole idea: querying DataFrames in place,
SQL aggregations and window functions, zero-copy results, the relational API, and reading
Parquet directly from disk. Everything here is CPU-only and runs in a fresh kernel.

### Example 1 — Query a pandas DataFrame in place (replacement scan)

In [ ]:
import pandas as pd

sales = pd.DataFrame(
    {
        "region": ["US", "US", "EU", "EU", "US", "EU"],
        "product": ["A", "B", "A", "B", "A", "A"],
        "units": [10, 5, 7, 3, 4, 8],
        "price": [2.5, 9.0, 2.5, 9.0, 2.5, 2.5],
    }
)

# `sales` (a Python variable) is used directly in FROM — no load step. DuckDB reads it via Arrow.
con = duckdb.connect()  # an explicit in-process connection
result = con.sql("""
    SELECT region,
           SUM(units)            AS total_units,
           SUM(units * price)    AS revenue
    FROM sales
    GROUP BY region
    ORDER BY revenue DESC
""")
result.show()

# Results are zero-copy convertible back to a DataFrame.
print(type(result.df()))
result.df()

### Example 2 — Window functions and the relational (Python) API

In [ ]:
# A window function: each region's revenue as a share of the grand total, no self-join needed.
con.sql("""
    SELECT region, product, units * price AS revenue,
           ROUND(100 * (units * price) / SUM(units * price) OVER (), 1) AS pct_of_total
    FROM sales
    ORDER BY revenue DESC
    LIMIT 4
""").show()

# The same kind of work without writing a SQL string — chain a lazy relation, then materialize.
rel = (
    con.table("sales")              # 'sales' is still visible as a replacement-scan table
       .filter("region = 'US'")
       .aggregate("product, sum(units) AS units, sum(units*price) AS revenue")
       .order("revenue DESC")
)
print("US breakdown by product:")
rel.show()

### Example 3 — Read Parquet directly from disk (out-of-core shape)

DuckDB's real strength is querying files it never fully loads. We write a Parquet file, then run
SQL straight against the *path* — DuckDB pushes the `WHERE`/projection into the scan and reads
only the columns and row-groups it needs.

In [ ]:
import os, tempfile

tmp = tempfile.mkdtemp()
path = os.path.join(tmp, "sales.parquet")

# Write the table out as Parquet entirely from SQL (COPY), then query the file by path.
con.sql(f"COPY sales TO '{path}' (FORMAT parquet)")

con.sql(f"""
    SELECT region, COUNT(*) AS rows, SUM(units) AS units
    FROM read_parquet('{path}')   -- or just FROM '{path}'
    WHERE units >= 5
    GROUP BY region
    ORDER BY region
""").show()

# DuckDB infers schema from the file; DESCRIBE shows what it found.
con.sql(f"DESCRIBE SELECT * FROM '{path}'").show()

### Example 4 — Persisted database and a remote read (gated)

A file-backed connection persists tables across runs. Reading remote data needs the `httpfs`
extension and network access, so it's gated behind an env check — the notebook still runs
end-to-end without it, while showing the call shape.

In [ ]:
# Persistent, single-file database — survives process restarts; supports larger-than-memory ops.
dbfile = os.path.join(tmp, "warehouse.duckdb")
with duckdb.connect(dbfile) as pcon:
    pcon.sql("CREATE TABLE IF NOT EXISTS sales AS SELECT * FROM read_parquet('%s')" % path)
    n = pcon.sql("SELECT COUNT(*) FROM sales").fetchone()[0]
    print(f"persisted {n} rows to {os.path.basename(dbfile)}")

# Remote read over HTTP/S3 — gated so the notebook executes offline.
if os.getenv("DUCKDB_RUN_REMOTE"):
    con.sql("INSTALL httpfs; LOAD httpfs;")
    url = "https://blobs.duckdb.org/data/Star_Trek-Season_1.parquet"
    print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{url}')").fetchall())
else:
    print("Set DUCKDB_RUN_REMOTE=1 (and have httpfs + network) to run the remote read.")
    print("Shape:  con.sql(\"INSTALL httpfs; LOAD httpfs;\"); "
          "con.sql(\"SELECT * FROM read_parquet('s3://bucket/key.parquet')\")")

con.close()

## 6. Gotchas & Pitfalls

- **It's OLAP, not OLTP.** DuckDB is built for big scans/aggregations, not many small concurrent
  writes or single-row lookups. One writer at a time; don't use it as your app's transactional
  database.
- **Single writer, multiple readers.** A persistent `.duckdb` file can't be opened for writing by
  two processes at once. Concurrent *reads* are fine; concurrent writers will error. For
  parallelism inside one process, DuckDB already multi-threads a single query.
- **Replacement scan needs the variable in scope.** `SELECT * FROM df` resolves `df` from the
  calling Python frame. If it's not visible (different module/function, or named after a real
  table), you'll get a "table does not exist" error or shadow a real table. Use
  `con.register("name", df)` to bind explicitly.
- **`.df()` materializes everything.** Pulling a billion-row result into pandas defeats the
  point and may blow up RAM. Aggregate/limit in SQL first; only convert the small final result.
- **Results are lazy relations.** `con.sql(...)` returns a relation that hasn't executed yet;
  it runs when you call `.show()`, `.df()`, `.fetchall()`, etc. A relation that closes over a
  connection is invalid after the connection closes.
- **In-memory connections are ephemeral.** `connect()` with no path keeps everything in RAM and
  vanishes when the connection closes — fine for scratch work, but use a file path to persist or
  to enable disk spilling for larger-than-memory queries.
- **CSV parsing isn't magic.** `read_csv_auto` sniffs types from a sample; weird quoting, mixed
  types, or odd date formats may need explicit options (`types=`, `dateformat=`, `sample_size=-1`
  to scan the whole file). Parquet, being typed, sidesteps all of this.
- **Extensions need install + load.** `httpfs`, `spatial`, `json`, etc. require
  `INSTALL x; LOAD x;` once per database (install is cached on disk, load is per session) and,
  for remote ones, network access.

## 7. When to Use vs Alternatives

| Option | Best at | Reach for it instead of DuckDB when… |
| --- | --- | --- |
| **DuckDB** | Local/medium analytical SQL over files and DataFrames; zero-setup OLAP | — |
| **SQLite** | Embedded *transactional* (OLTP) storage, many small reads/writes, app state | You need row-oriented point lookups and durable single-row writes, not aggregations |
| **pandas** | Small in-memory data, rich Python ecosystem (sklearn/SciPy), ad-hoc munging | Data is small and you want imperative DataFrame code, not SQL |
| **Polars** | Fast multi-core DataFrame *API*, lazy expressions, streaming | You prefer a method-chaining DataFrame API to SQL (DuckDB ↔ Polars interop is zero-copy) |
| **Postgres** | Shared, concurrent, multi-writer transactional database with a server | You need many concurrent writers, durability, and client/server access control |
| **Spark / Trino / BigQuery** | Distributed queries over TB–PB across a cluster | The data genuinely doesn't fit on one machine and needs horizontal scale |

**Rules of thumb.** Reaching for `GROUP BY`/joins over Parquet/CSV on one machine → DuckDB. App
needs a durable embedded store with frequent small writes → SQLite. Heavy method-chaining
DataFrame transforms with no SQL → Polars (and hand frames to DuckDB over Arrow when SQL is
cleaner). Multi-user concurrent writes → Postgres. Cluster-scale → Spark/Trino/BigQuery. DuckDB
and Polars are complements, not rivals — both sit on Arrow and pass data between each other for
free, so use SQL where SQL is clearer and expressions where they are.

## 8. Resources

- **Official docs** — https://duckdb.org/docs/ (the "Guides" and SQL reference are excellent and
  task-oriented).
- **Python API guide** — https://duckdb.org/docs/stable/clients/python/overview (connections,
  replacement scans, the relational API, DataFrame interop).
- **"Why DuckDB"** — https://duckdb.org/why_duckdb (the design rationale: in-process, columnar,
  vectorized, and how it compares to SQLite/Postgres/Spark).
- **Friendlier SQL features** — https://duckdb.org/docs/stable/sql/dialect/friendly_sql
  (quality-of-life extensions like `GROUP BY ALL`, `SELECT * EXCLUDE (...)`, trailing commas).
- **Modern Data Stack in a Box / blog** — https://duckdb.org/2022/10/12/modern-data-stack-in-a-box.html
  (a worked example of DuckDB as the engine of a local analytics stack).

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def bytes_scanned(row_groups, widths, projected, predicate=None, layout="column"):
    """How much of a file each storage layout actually has to read for one query."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE